In [ ]:
import pyroomacoustics as pra
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
RIR_DIR = PROJECT_ROOT / "rirs"
RIR_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
room_dim = [3, 4, 2]
length_of_rirs = 512
fs = 16000
position_of_reference_microphone = [1.5, 1, 1]
position_of_canceling_loudspeaker = [1.5, 2.5, 1]
position_of_error_microphone = [1.5, 3, 1]
rt60s = [0.15, 0.175, 0.2, 0.225, 0.25]
pri_path_filename = "Primary_path.mat"
pri_path_name = "Pz1"
sec_path_filename = "Secondary_path.mat"
sec_path_name = "S"

In [ ]:

for rt60 in rt60s:
    sub_dir = "rt60_" + str(rt60) + "s"
    save_dir = RIR_DIR / sub_dir
    e_absorption, max_order = pra.inverse_sabine(rt60, room_dim)
    room = pra.ShoeBox(p=room_dim, fs=fs, materials=pra.Material(e_absorption), max_order=max_order, )
    room.add_source(position_of_reference_microphone)
    room.add_source(position_of_canceling_loudspeaker)
    room.add_microphone(position_of_error_microphone)
    room.compute_rir()
    pri_path = np.expand_dims(room.rir[0][0][:length_of_rirs], axis=1)
    sec_path = np.expand_dims(room.rir[0][1][:length_of_rirs], axis=1)

    if not Path.exists(save_dir):
        Path.mkdir(save_dir)
    
    sio.savemat(str(save_dir/pri_path_filename), {pri_path_name: pri_path})
    sio.savemat(str(save_dir/sec_path_filename), {sec_path_name: sec_path})